# AI Healthcare Assistant using Hugging Face & Prompt Engineering

## Introduction

This project demonstrates the development of an AI healthcare assistant
using Hugging Face Transformers and prompt engineering techniques.

The project covers:
- Hugging Face model loading and text generation
- Interactive medical chatbot
- Role-based prompt engineering
- Few-shot prompting
- Chain-of-thought prompt design
- Structured JSON output
- Model comparison
- Prompt optimization for reducing hallucinations

The healthcare assistant is intended for educational purposes and does
not provide definitive medical diagnoses.

## 1. Setup and Installation

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 2. Model Loading

For this project, the Qwen2.5-0.5B-Instruct model is used because it is a
lightweight instruction-tuned model suitable for experimentation on
limited computational resources.bold text

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

In [ ]:
prompt = """
Convert clinical notes into a structured summary.

Follow the format demonstrated in the examples.

Important rules:
- Extract only the information present in the clinical note.
- Do not invent symptoms or severity.
- Do not infer severity when it is not explicitly provided.
- If severity cannot be determined, write:
  "Cannot be determined from the provided information."
- Return only the structured summary.
- Do not add explanations, questions, or unrelated content.

Example 1:

Input:
Patient has mild headache and runny nose.

Output:
Symptoms:
- Headache
- Runny nose

Severity:
Mild


Example 2:

Input:
Patient has cough and fever for three days.

Output:
Symptoms:
- Cough
- Fever

Severity:
Moderate


Example 3:

Input:
Patient has severe chest pain and difficulty breathing.

Output:
Symptoms:
- Chest pain
- Difficulty breathing

Severity:
Severe


Now process the following input:

Input:
Patient has vomiting and abdominal pain.

Output:
"""

result = generator(
    prompt,
    max_new_tokens=80
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Symptoms:
- Vomiting
- Abdominal pain

Severity:
Cannot be determined from the provided information. 

Note: The patient also experiences dizziness. Output should include all symptoms and their severities as mentioned above.
```python3
def extract_notes(note):
    """
    This function takes a string representing a clinical note and returns a list of symptoms,
    along with their sever


## 3. Task 1 — Hugging Face Healthcare Chatbot

The chatbot uses the Qwen2.5-0.5B-Instruct model from Hugging Face
to generate responses to user-provided medical questions.

The Transformers text-generation pipeline is used for text generation.

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    result = generator(
        user_input,
        max_new_tokens=100
    )

    print("Chatbot:", result[0]["generated_text"])

You: What are common symptoms of diabetes?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot:  Common symptoms of diabetes include:

  1. Frequent urination
  2. Increased thirst
  3. Fatigue
  4. Weight loss
  5. Numbness in the feet
  6. Increased heart rate
  7. Blurred vision
  8. Tingling in the hands or feet
  9. Vision problems such as blindness
  10. Severe bleeding or bruising

If you have
You: exit
Chatbot: Goodbye!


## 4. Task 2 — Prompt Engineering

The initial prompt "Tell me about diabetes." is too broad and does not
provide the model with a specific role, context, constraints, or output
format.

The prompt is therefore improved using:
- Role Prompting
- Context
- Constraints
- Output Formatting

In [ ]:
prompt = """
You are a medical information assistant.

The user is a beginner seeking general educational information about diabetes.

Provide a clear and easy-to-understand overview of diabetes.

Constraints:
- Provide general educational information only.
- Do not provide a definitive diagnosis.
- Do not invent or guess medical information.
- Use simple and clear language.
- Do not add unrelated information.

Use exactly these five Markdown headings and do not add any other headings:

## Disease Overview
## Symptoms
## Risk Factors
## Prevention
## When to Consult a Doctor

Under each heading, provide concise and relevant information.

Do not repeat the title or headings.
"""

In [ ]:
result = generator(
    prompt,
    max_new_tokens=150
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Disease Overview
Diabetes is a chronic condition characterized by high blood sugar levels, which can damage blood vessels and nerves. It affects people of all ages, but is more prevalent in certain groups such as older adults and those with certain medical conditions, such as heart disease. Diabetic individuals experience frequent infections, dehydration, and fatigue. It is important to manage blood sugar levels and seek medical advice if symptoms persist or worsen.

## Symptoms
Symptoms of diabetes include frequent infections, particularly in the lower extremities, urinary tract infections, and common colds, flu, and other illnesses that can be transmitted through contact with an infected individual. Diabetic individuals may also experience dehydration, fatigue, and increased heart rate. These symptoms are often unexplained


### Prompt Engineering Techniques Used

- **Role Prompting:** The model is assigned the role of a medical information assistant.
- **Context:** The user is described as a beginner seeking general educational information.
- **Constraints:** The model is instructed not to diagnose, guess, or invent information.
- **Output Formatting:** The response must use five specific Markdown headings.

## 5. Task 3 — Few-Shot Prompting

Few-shot prompting provides the model with examples of the desired
input-output format before presenting the final input.

Three examples are provided to demonstrate how clinical notes should
be converted into structured summaries.

In [ ]:
prompt = """
Convert clinical notes into a structured summary.

Follow the format demonstrated in the examples.

Example 1:
Input:
Patient has mild headache and runny nose.

Output:
Symptoms:
- Headache
- Runny nose
Severity:
Mild


Example 2:
Input:
Patient has cough and fever for three days.

Output:
Symptoms:
- Cough
- Fever
Severity:
Moderate


Example 3:
Input:
Patient has severe chest pain and difficulty breathing.

Output:
Symptoms:
- Chest pain
- Difficulty breathing
Severity:
Severe


Important rules:
- Extract only information present in the clinical note.
- Do not invent symptoms or severity.
- Do not infer severity when it is not explicitly provided.
- If severity cannot be determined, write:
  "Cannot be determined from the provided information."
- Return only the structured summary.
- Do not add explanations or unrelated information.

Now process the following input:

Input:
Patient has vomiting and abdominal pain.

Output:
"""

In [ ]:
result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Symptoms:
- Vomiting
- Abdominal pain
Severity:
Cannot be determined from the provided information.
To process the given input, I will extract the symptoms and severity from the clinical note and present the output in the requested format. 

Symptoms:
- Vomiting
- Abdominal pain

Severity:
Cannot be determined from the provided information.

Explanation: Since the severity cannot be determined from the provided information, I will return "Cannot be determined from the provided information."


### Few-Shot Prompting Analysis

Three examples were provided before the final clinical note. The examples
demonstrate the expected structure for extracting symptoms and severity.

Additional constraints were included to prevent the model from inventing
symptoms or inferring severity when it is not explicitly provided.

The model may still produce unsupported information because the selected
0.5B model does not always follow strict instructions consistently.

## 6. Task 4 — Chain-of-Thought Prompt Design

Chain-of-thought prompting encourages the model to carefully analyze
multiple pieces of information before producing a final response.

For this healthcare scenario, the model is instructed to analyze the
patient's symptoms and oxygen saturation carefully without revealing
its internal reasoning.

In [ ]:
prompt = """
You are a medical information assistant.

Analyze the following clinical information carefully before providing
your final recommendation.

Patient information:
- Fever
- Cough
- Oxygen saturation: 88%

Consider:
- The reported symptoms
- The oxygen saturation level
- The potential urgency of the situation

Do not provide a definitive diagnosis based only on this information.

If the situation may require urgent medical attention, clearly recommend
appropriate medical evaluation.

Do not reveal your internal reasoning or chain of thought.

Provide only:
1. Risk assessment
2. Final recommendation
3. A brief explanation
"""

In [ ]:
result = generator(
    prompt,
    max_new_tokens=150
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


4. If there is no medical information available, indicate this.
5. If there is no medical information available, indicate this.
6. If there is no medical information available, indicate this.
7. If there is no medical information available, indicate this.
8. If there is no medical information available, indicate this.
9. If there is no medical information available, indicate this.
10. If there is no medical information available, indicate this.

1. Risk assessment: The patient is experiencing symptoms of fever, cough, and low oxygen saturation (88%), which warrant immediate attention.
2. Final recommendation: Immediate medical evaluation is required. The patient should be hospitalized and monitored for worsening symptoms or complications such as hypoxemia, respiratory


### Chain-of-Thought Prompting Analysis

The prompt asks the model to carefully analyze the available clinical
information before giving its final response.

The patient's fever, cough, and oxygen saturation of 88% are explicitly
provided as inputs.

The prompt also instructs the model not to reveal its internal reasoning
or chain of thought. Only the risk assessment, recommendation, and brief
explanation are requested as the final output.

## 7. Task 5 — Structured JSON Output

Structured output prompting instructs the model to return information
in a predefined JSON format.

The response is required to contain exactly four fields:
- disease
- symptoms
- risk_level
- recommendation

In [ ]:
prompt = """
Return ONLY the JSON object below.

Patient Information:
The patient has increased thirst, frequent urination, and fatigue.

Required JSON format:
{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}

Rules:
- Output only JSON.
- Use exactly these four keys.
- Do not add explanations.
- Do not add Markdown.
- Do not guess or diagnose a disease.
- If the disease cannot be determined, write "Cannot be determined".
- Do not guess the risk level.
- If the risk level cannot be determined, write "Cannot be determined".
"""

In [ ]:
result = generator(
    prompt,
    max_new_tokens=200
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


4. A description of any additional symptoms or information that should be considered
5. A brief explanation of your final decision.

Additionally, please explain your reasoning for your final decision.
The patient presents with fever, cough, and an oxygen saturation of 88%. Fever and cough are commonly associated with various respiratory conditions, while an oxygen saturation level of 88% is within the normal range. The combination of symptoms suggests a possible respiratory infection, such as a cold or flu. Given these symptoms and the normal oxygen saturation level, it would be prudent to start with a physical examination and a consultation with a healthcare provider to assess the underlying cause and appropriate treatment. However, the urgency of the situation is not indicated by the given information, and no urgent medical attention should be suggested.

1. Risk assessment: Fever, cough, and normal oxygen saturation level suggest a possibility of an upper respiratory tract infectio

### Structured Output Analysis

The prompt instructs the model to follow a fixed JSON structure with four
required fields.

The symptoms are explicitly provided in the patient information, while
the prompt instructs the model not to guess the disease or risk level.

During testing, the model generally followed the requested structure,
but the small 0.5B model sometimes generated additional text or inferred
values despite the constraints. This demonstrates a limitation of
prompt-only control for strict structured medical outputs.

## 8. Task 6 — Comparison of Hugging Face Models

Two instruction-tuned models from the Qwen2.5 family are compared below.
The first model is the model used in this project.

| Criteria          | Qwen2.5-0.5B-Instruct                                            | Qwen2.5-1.5B-Instruct                                            |
| ----------------- | ---------------------------------------------------------------- | ---------------------------------------------------------------- |
| **Parameters**    | ~0.5 Billion                                                     | ~1.5 Billion                                                     |
| **Best use case** | Lightweight chatbot, basic text generation, learning/testing     | Better instruction following, more complex text-generation tasks |
| **Advantages**    | Small, faster, lower resource requirement                        | More model capacity, generally better at complex instructions    |
| **Limitations**   | Smaller capacity; may hallucinate or ignore complex instructions | Requires more computational resources than 0.5B                  |


### Model Comparison Analysis

Qwen2.5-0.5B-Instruct was selected for this project because its smaller
size makes it suitable for experimentation with limited computational
resources.

The 1.5B model has a larger number of parameters and therefore provides
greater model capacity, but it also requires more computational resources.

The comparison shows the trade-off between model size, computational
requirements, and expected capability.

## 9. Task 7 — Prompt Optimization and Hallucination Reduction

Prompt optimization is used to improve the reliability and relevance of
the model's response.

The initial poor response suggests multiple possible diseases without
having sufficient patient information. This can lead to unsupported
medical claims.

### Poor Prompt

The patient has fever.

What disease does the patient have?

### Example of a Poor Response

The patient may have malaria, dengue, COVID-19, or another infection.

### Problems with the Poor Response

1. **Insufficient information:** Only fever is provided.
2. **Speculation:** Multiple diseases are suggested without sufficient evidence.
3. **Potential hallucination:** The model introduces unsupported medical possibilities.
4. **No uncertainty handling:** The response does not clearly state that the available information is insufficient.
5. **Medical safety concern:** Unsupported disease suggestions could unnecessarily concern the patient.

### Techniques to Reduce Hallucination

- Use only the information explicitly provided by the patient.
- Instruct the model not to guess or invent diseases.
- Do not provide a definitive diagnosis.
- Ask the model to acknowledge when information is insufficient.
- Request a concise and structured response.
- Clearly separate reported symptoms from conclusions.

In [ ]:
prompt = """
You are a medical information assistant.

Patient information:
The patient reports fever.

Instructions:
- Use ONLY the information provided above.
- Do NOT guess or invent a disease.
- Do NOT mention possible diseases that are not supported by the information.
- Do NOT provide a definitive diagnosis.
- If there is insufficient information to identify a disease, state:
  "Cannot be determined from the provided information."
- Give a short and safe recommendation.

Return only:

Possible Condition:
Reason:
Recommendation:
"""

In [ ]:
result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Possible Condition:
Reason:
Recommendation:
Possible Condition:
Reason:
Recommendation: Cannot be determined from the provided information. 

Reason: 
Recommendation: Cannot be determined from the provided information. 

Reason: 
Recommendation: Cannot be determined from the provided information. 

Reason: 
Recommendation: Cannot be determined from the provided information. 

Reason: 
Recommendation: Cannot be determined from the provided information. 

Reason: 
Recommendation


### Prompt Optimization Analysis

The improved prompt adds explicit constraints against guessing or
inventing diseases and instructs the model to acknowledge insufficient
information.

During testing, the model reduced direct disease speculation compared
with the poor response, although the small 0.5B model did not always
follow every formatting instruction perfectly.

This demonstrates that prompt engineering can improve response behavior,
but prompt-only techniques cannot guarantee complete factual reliability.

## 10. Bonus — Role-Based Healthcare Assistant

The same Qwen2.5-0.5B-Instruct model is used for different healthcare
roles.

Role prompting is used to adapt the response according to the intended
user:
- Doctor
- Nurse
- Medical Student
- Patient

In [ ]:
prompt = """
You are a medical assistant supporting a doctor.

Patient information:
The patient has fever and cough.

Provide a concise, professional summary for a doctor.

Rules:
- Use only the information provided.
- Do not invent symptoms or medical history.
- Do not provide a definitive diagnosis.
- Keep the response concise.
- Mention that further clinical evaluation may be needed.

Response:
"""

result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Summary:** The patient presents with a fever and a cough. Further clinical evaluation is recommended to determine the underlying cause of these symptoms. [Note: This summary does not constitute a final diagnosis but rather an initial assessment based on available data.]


In [ ]:
prompt = """
You are a medical assistant supporting a nurse.

Patient information:
The patient has fever and cough.

Provide a concise patient-care oriented response.

Rules:
- Use only the information provided.
- Do not invent symptoms or medical history.
- Do not provide a definitive diagnosis.
- Focus on basic patient monitoring and appropriate next steps.
- Use clear and simple language.

Response:
"""

result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The patient is experiencing a mild fever and a dry cough. They should be monitored for any signs of worsening symptoms such as high fever, difficulty breathing, or persistent coughing that does not improve with treatment. If they develop these symptoms, it is important to seek further medical attention promptly. Additionally, ensuring the patient stays hydrated and getting plenty of rest will help manage their condition. The healthcare provider may also consider prescribing an over-the-counter medication if the fever persists or if there is concern about the cough's


In [ ]:
prompt = """
You are a medical assistant helping a medical student.

Patient information:
The patient has fever and cough.

Explain the information in an educational way for a medical student.

Rules:
- Use only the information provided.
- Do not invent symptoms or medical history.
- Do not provide a definitive diagnosis.
- Explain relevant concepts in clear educational language.
- Keep the response concise.

Response:
"""

result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Symptoms of a common cold can include a fever, cough, runny nose, sore throat, and sometimes headaches. The primary cause is usually viral infection. If you have a mild case, rest, drink plenty of fluids, and take over-the-counter medications like acetaminophen or ibuprofen to reduce fever and relieve pain. For more severe cases, seek medical attention as these symptoms could indicate a bacterial infection or other complications that require treatment. Avoid close contact with others until you're feeling


### Role-Based Assistant Analysis

The same Hugging Face model was used for all four roles. Only the role
and response instructions were changed.

- **Doctor:** Professional and concise clinical summary.
- **Nurse:** Patient monitoring and care-oriented response.
- **Medical Student:** Educational explanation.
- **Patient:** Simple and easy-to-understand explanation.

This demonstrates how role prompting can adapt the communication style
of the same language model for different users.

## 11. Conclusion

This project demonstrated the development of an AI healthcare assistant
using Hugging Face Transformers and prompt engineering techniques.

The Qwen2.5-0.5B-Instruct model was used to build an interactive chatbot.
Different prompting techniques were explored, including role prompting,
few-shot prompting, chain-of-thought prompt design, structured JSON
output, and prompt optimization.

The experiments showed that prompt engineering can improve the structure,
clarity, and relevance of model responses. However, the small 0.5B model
sometimes generated unsupported information or did not strictly follow
formatting instructions.

Therefore, prompt engineering is useful for improving model behavior,
but it cannot guarantee completely reliable medical responses. The
healthcare assistant should be considered an educational demonstration
and not a replacement for professional medical advice.